In [22]:
import pandas as pd

Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

A. 30 ngày

B. 90 ngày

**C. 180 ngày**

D. 365 ngày

In [33]:
orders = pd.read_csv("data/orders.csv", parse_dates=["order_date"])

# Created
# orders = orders[orders['order_status'] != 'cancelled']

# No Created
valid_statuses = ['delivered', 'shipped', 'paid', 'returned']
orders = orders[orders['order_status'].isin(valid_statuses)].copy()

orders = orders.sort_values(['customer_id', 'order_date'])

orders['gap_days'] = orders.groupby('customer_id')['order_date'].diff().dt.days

valid_customers = orders.groupby('customer_id').filter(lambda x: len(x) > 1)

gaps = valid_customers['gap_days'].dropna()

median_gap = gaps.median()

print(f"{median_gap:.0f} ngày")

158 ngày


Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

A. Premium

B. Performance

C. Activewear

**D. Standard**

In [6]:
products = pd.read_csv("data/products.csv")

products['gpm'] = (products['price'] - products['cogs']) / products['price']

segment_performance = products.groupby('segment')['gpm'].mean().sort_values(ascending=False)

print(segment_performance)

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gpm, dtype: float64


Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

A. defective

**B. wrong_size**

C. changed_mind

D. not_as_described

In [7]:
returns = pd.read_csv("data/returns.csv")
products = pd.read_csv("data/products.csv")

merged_df = returns.merge(products[['product_id', 'category']], on='product_id')

streetwear_returns = merged_df[merged_df['category'] == 'Streetwear']

reason_counts = streetwear_returns['return_reason'].value_counts()

print(reason_counts)


return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

A. organic_search

B. paid_search

**C. email_campaign**

D. social_media

In [9]:
web_traffic = pd.read_csv("data/web_traffic.csv")

source_bounce_rate = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()

print(source_bounce_rate)

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

A. 12%

B. 25%

**C. 39%**

D. 54%

In [26]:
order_items = pd.read_csv("data/order_items.csv", dtype={'promo_id': str, 'promo_id_2': str})

promo_percentage = order_items['promo_id'].notnull().mean() * 100

print(f"{promo_percentage:.0f}%")

39%


Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

**A. 55+**

B. 25–34

C. 35–44

D. 45–54

In [15]:
customers = pd.read_csv("data/customers.csv")
orders = pd.read_csv("data/orders.csv")

valid_customers = customers[customers['age_group'].notnull()]

merged_df = orders.merge(valid_customers[['customer_id', 'age_group']], on='customer_id')

group_stats = merged_df.groupby('age_group').agg(
    total_orders=('order_id', 'count'),
    total_customers=('customer_id', 'nunique')
)
group_stats['avg_orders_per_customer'] = group_stats['total_orders'] / group_stats['total_customers']

print(group_stats.sort_values('avg_orders_per_customer', ascending=False))

           total_orders  total_customers  avg_orders_per_customer
age_group                                                        
55+               72760            10010                 7.268731
45-54            124138            17193                 7.220264
35-44            170368            23642                 7.206159
25-34            190622            26802                 7.112230
18-24             89057            12599                 7.068577


Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

A. West

B. Central

**C. East**

D. Cả ba vùng có doanh thu xấp xỉ bằng nhau

In [27]:
order_items = pd.read_csv("data/order_items.csv", dtype={'promo_id': str, 'promo_id_2': str})
orders = pd.read_csv("data/orders.csv")
geography = pd.read_csv("data/geography.csv")

order_items['revenue'] = order_items['quantity'] * order_items['unit_price']

merged_df = order_items.merge(orders[['order_id', 'zip']], on='order_id') \
                       .merge(geography[['zip', 'region']], on='zip')

region_revenue = merged_df.groupby('region')['revenue'].sum().sort_values(ascending=False)

print(region_revenue)

region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: revenue, dtype: float64


Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

**A. credit_card**

B. cod

C. paypal

D. bank_transfer

In [18]:
orders = pd.read_csv("data/orders.csv")

cancelled_orders = orders[orders['order_status'] == 'cancelled']

payment_counts = cancelled_orders['payment_method'].value_counts()

print(payment_counts)

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

**A. S**

B. M

C. L

D. XL

In [28]:
returns = pd.read_csv("data/returns.csv")
order_items = pd.read_csv("data/order_items.csv", dtype={'promo_id': str, 'promo_id_2': str})
products = pd.read_csv("data/products.csv")

returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')
items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id')

target_sizes = ['S', 'M', 'L', 'XL']
returns_filtered = returns_with_size[returns_with_size['size'].isin(target_sizes)]
items_filtered = items_with_size[items_with_size['size'].isin(target_sizes)]

returns_count = returns_filtered['size'].value_counts()
items_count = items_filtered['size'].value_counts()

return_rate = (returns_count / items_count).sort_values(ascending=False)

print(return_rate)

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64


Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

A. 1 kỳ (trả một lần)

B. 3 kỳ

**C. 6 kỳ**

D. 12 kỳ

In [21]:
payments = pd.read_csv("data/payments.csv")

avg_payment_per_installment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

print(avg_payment_per_installment)

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
